## Benchmark: Note Detection

This is the 2nd of 3 component benchmarking we'll be conducting to test the general performance of the TuneBuddy system.

1. Pitch detection
2. Note detection
3. App usefulness

In [1]:
# autoreload when modules edited
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

from app_logic.midi.MidiData import MidiData
from app_logic.user.ds.UserData import UserData
# from app_logic.user.AudioRecorder import AudioRecorder
# from app_logic.user.AudioPlayer import AudioPlayer

from algorithms.PitchDetector import PitchDetector
from algorithms.NoteDetector import NoteDetector
from algorithms.StringEditor import StringEditor
from algorithms.Config import Config

In [51]:
# test pitch detection accuracy
import pretty_midi
import soundfile as sf

SF_PATH = "../resources/MuseScore_General.sf3"
INPUT_MIDI_PATH = "../resources/scores/fugue.mid"
OUTPUT_PATH = "../resources/scores/midi-to-audio/fugue.wav"

MY_AUDIO = "../resources/recordings/bach_fugue_recording.mp3"

W2 = 21

config = {
    'sr': 44100,    # sample rate
    'w1': 1024 * 2,  # frame size
    'h1': 128,       # hop size
    'fmin': 196.0,
    'fmax': 3000.0,
    'tuning': 440.0,
    'unv_thresh': 0.9, # if unvoiced_prob > unv_thresh, consider the frame unvoiced

    # --- NOTE DETECTION PARAMETERS ---
    'w2': W2, # frame size (NOTE: should always be odd)
    'h2': W2-2, # hop size
    'pitch_thresh': 0.5,
    'slope_thresh': 0.75 / W2,
    'unv_ratio': 0.8, # proportion of unvoiced pitches in a window to consider the window unvoiced

    # --- STRING EDIT PARAMETERS ---
    'ins_cost': 1.5,
    'del_cost': 2,
    'sub_cost': 1,
    'tolerance': 1,
    # tiger-mom parameter
    'tiger_level': 1
}
config = Config(**config)

# synthesize midi to audio
# mid = pretty_midi.PrettyMIDI(INPUT_MIDI_PATH)
# audio = mid.fluidsynth(synthesizer=SF_PATH, fs=44100)
# sf.write(OUTPUT_PATH, audio, 44100)

# load and detect pitches
mid1 = MidiData(INPUT_MIDI_PATH)

pd, nd = PitchDetector(config), NoteDetector(config)
us1 = UserData(pd, nd)
us1.load_audio(MY_AUDIO)

Loading score file: ../resources/scores/fugue.mid (ext: .mid)
Handling MIDI file...


100%|██████████| 4498/4498 [00:02<00:00, 2068.15it/s]


### Re-running note detection: Different window sizes

Takeaway: Window size doesn't matter much as long as slope is accurately adjusted.

In [ ]:
# retry note detection until we minimize string edit distance
se = StringEditor(config)

# resize midi to match user audio length

mid1.resize(new_length=us1.audio_data.get_length())

# find maximum window size for note detection based on minimum note length
min_note_sec = mid1.note_data.get_minimum_note_length()
max_window = int((config.sr / config.h1) * min_note_sec)
max_window = max_window if max_window % 2 == 1 else max_window + 1 # ensure parity

# try all windows up to max_window (odd only)
poss_windows = [w for w in range(3, max_window, 2)]

print(f"min note length: {min_note_sec:.3f} sec, max window size is {max_window} frames")
print(f"trying {len(poss_windows)} window sizes...")

best_cost = float('inf')
best_window = max_window

for w in poss_windows:
    # update this stuff
    config.w2 = w
    config.h2 = w-2 # hop size
    config.slope_thresh = 0.75 / w
    us1.detect_notes()
    algn = se.string_edit(
        us1.note_data, 
        mid1.note_data
    )
    n = len(algn.mistakes)
    print(f"    w2={w}, slope={config.slope_thresh:.4f}, string edit cost: {n}")
    if n < best_cost:
        best_cost = n
        best_window = w


# rerun note detection with the best one
config.w2 = best_window
config.h2 = best_window-2 # hop size
config.slope_thresh = 0.5 / best_window
us1.detect_notes()

algn = se.string_edit(
    us1.note_data, 
    mid1.note_data
)
print(f"best found!\n --> window size: {best_window}, string edit cost: {best_cost}")

min note length: 0.197 sec, max window size is 67 frames
trying 32 window sizes...
    w2=3, slope=0.2500, string edit cost: 11
    w2=5, slope=0.1500, string edit cost: 11
    w2=7, slope=0.1071, string edit cost: 11
    w2=9, slope=0.0833, string edit cost: 11
    w2=11, slope=0.0682, string edit cost: 11
    w2=13, slope=0.0577, string edit cost: 11
    w2=15, slope=0.0500, string edit cost: 11
    w2=17, slope=0.0441, string edit cost: 11
    w2=19, slope=0.0395, string edit cost: 11
    w2=21, slope=0.0357, string edit cost: 11
    w2=23, slope=0.0326, string edit cost: 11
    w2=25, slope=0.0300, string edit cost: 11
    w2=27, slope=0.0278, string edit cost: 11
    w2=29, slope=0.0259, string edit cost: 11
    w2=31, slope=0.0242, string edit cost: 11
    w2=33, slope=0.0227, string edit cost: 11
    w2=35, slope=0.0214, string edit cost: 11
    w2=37, slope=0.0203, string edit cost: 11
    w2=39, slope=0.0192, string edit cost: 11
    w2=41, slope=0.0183, string edit cost: 11
 

In [53]:
%gui qt

import sys
from PyQt6.QtWidgets import QApplication
from PyQt6.QtCore import QCoreApplication

sys.path.append('../')
from ui.ScorePlot import RunScorePlot

if __name__ == '__main__':
    if not QCoreApplication.instance():
        app = QApplication(sys.argv)
    else:
        app = QCoreApplication.instance()

    vis = RunScorePlot(user_data=us1, midi_data=mid1, app=app)

Loading MIDI data into ScorePlot...
Loading UserData into ScorePlot...


In [11]:
print(config)

Config
---
   sr=44100, w1=8192, h1=128, fmin=196.0, fmax=3000.0, tuning=440.0, unv_thresh=0.9,
   w2=3, h2=1, pitch_thresh=0.5, slope_thresh=0.167, unv_ratio=0.8,
   ins_cost=1.5, del_cost=2, sub_cost=1, tolerance=1, tiger_level=1
